# Multi-Feature Signal Construction

**Goal:** Combine multiple normalized features into a single tradeable signal and compare performance against individual feature signals.

**Three modes demonstrated:**
1. `binary_threshold` — trade when |combined signal| exceeds a threshold
2. `proportional` — fractional sizing proportional to signal strength  
3. `ensemble_vote` — each feature votes +1/−1; majority wins

**Workflow:** Features → Z-score normalize → Combine → Signal → IC vs. individual features

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.data.loader import FXDataLoader
from src.features import (
    compute_log_returns, atr, causal_hp_trend, hp_trend_slope,
    ma_spread_on_trend, ma_crossover_signal, trend_deviation_from_ma,
    trend_strength,
    normalize_features,
    combine_features, signal_to_position, SignalConfig,
    compute_ic, compute_forward_returns,
)

plt.style.use('seaborn-v0_8-darkgrid')
print('Imports OK')

## 1. Load data and build feature pipeline

In [ ]:
loader = FXDataLoader('../data/raw')
df = loader.load('USDJPY_10yr_1h_dukascopy')
close = df['close']
high = df['high']
low = df['low']
log_ret = compute_log_returns(close)
atr_s = atr(high, low, close, window=20)

print('Computing causal HP trend...')
trend = causal_hp_trend(close, lamb=3.9e9, window=500)
slope = hp_trend_slope(trend)
print('Done.')

# Mean-reversion features are negated so positive = expect positive return
raw_features = {
    'ma_spread_inv':     -ma_spread_on_trend(trend, t1=72, t2=240, atr_series=atr_s),
    'trend_dev_inv':     -trend_deviation_from_ma(trend, window=240, atr_series=atr_s),
    'crossover_inv':     -ma_crossover_signal(trend, t1=72, t2=240).replace({np.nan: 0.0}),
    'trend_strength':     trend_strength(trend, atr_s),
}

print('\nRaw feature stats:')
pd.DataFrame(raw_features).describe().round(3)

## 2. Normalize features

In [ ]:
NORM_WINDOW = 252 * 24  # 1 year

features_norm = normalize_features(raw_features, method='zscore', window=NORM_WINDOW)

print('Normalized feature stats (should be ~mean=0, std=1):')
pd.DataFrame(features_norm).dropna().describe().round(3)

## 3. Combine features — all three modes

In [ ]:
HORIZON = 24
fwd_ret = compute_forward_returns(close, HORIZON)

modes = ['binary_threshold', 'proportional', 'ensemble_vote']
threshold = 0.3

combined_signals = {}
positions = {}
ics = {}

for mode in modes:
    cfg = SignalConfig(mode=mode, threshold=threshold)
    sig = combine_features(features_norm, cfg)
    pos = signal_to_position(sig, mode=mode, threshold=threshold)
    combined_signals[mode] = sig
    positions[mode] = pos

    ic_result = compute_ic(sig.dropna(), fwd_ret)
    ics[mode] = ic_result.abs_ic
    print(f'{mode:22s}  |IC| = {ic_result.abs_ic:.4f}  p={ic_result.p_value:.4f}  n={ic_result.n_obs:,}')

# Also compute IC for individual features
print('\nIndividual feature ICs (for comparison):')
for name, series in features_norm.items():
    ic_r = compute_ic(series.dropna(), fwd_ret)
    ics[name] = ic_r.abs_ic
    print(f'{name:22s}  |IC| = {ic_r.abs_ic:.4f}  p={ic_r.p_value:.4f}')

## 4. IC comparison: combined vs. individual

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

labels = list(ics.keys())
values = list(ics.values())
colors = ['royalblue' if l in modes else 'steelblue' for l in labels]

bars = ax.bar(range(len(labels)), values, color=colors, edgecolor='white')
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=30, ha='right', fontsize=9)
ax.set_ylabel('|IC| (Spearman rank correlation)')
ax.set_title(f'Multi-Feature Signal IC vs. Individual Feature IC  (H={HORIZON})', fontsize=12)
ax.axhline(0.03, color='red', linestyle='--', linewidth=1, label='IC threshold = 0.03')
ax.legend()

# Label bars
for bar, val in zip(bars, values):
    if not np.isnan(val):
        ax.text(
            bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.001,
            f'{val:.3f}', ha='center', va='bottom', fontsize=8,
        )

plt.tight_layout()
plt.savefig('../reports/signal_ic_comparison.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: reports/signal_ic_comparison.png')

## 5. Signal overlay on price

In [ ]:
sample = slice(-2000, None)  # last 2000 bars

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

axes[0].plot(close.iloc[sample], lw=0.8, color='black', label='USDJPY close')
axes[0].set_ylabel('Price')
axes[0].set_title('USDJPY Close Price (last 2000 bars)')
axes[0].legend()

sig_sample = combined_signals['binary_threshold'].iloc[sample]
axes[1].fill_between(sig_sample.index, sig_sample, 0,
                     where=sig_sample > 0, color='green', alpha=0.4, label='long signal')
axes[1].fill_between(sig_sample.index, sig_sample, 0,
                     where=sig_sample < 0, color='red', alpha=0.4, label='short signal')
axes[1].axhline(threshold, color='green', linestyle='--', lw=0.8, label=f'+threshold={threshold}')
axes[1].axhline(-threshold, color='red', linestyle='--', lw=0.8, label=f'-threshold={threshold}')
axes[1].axhline(0, color='black', lw=0.5)
axes[1].set_ylabel('Combined Signal')
axes[1].set_title('Binary Threshold Combined Signal')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig('../reports/signal_overlay.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: reports/signal_overlay.png')